<div style="background:linear-gradient(135deg,#51247a 0%,#7a3fa0 100%);padding:22px 26px;border-radius:10px;margin-bottom:1.2rem;border-bottom:4px solid #f0a500;">
<div style="font-size:2.2rem;margin-bottom:6px;">🧹</div>
<div style="color:white;font-size:1.5rem;font-weight:700;">Text Cleaner</div>
<div style="color:rgba(255,255,255,.8);font-size:.92rem;margin-top:4px;">Remove and replace patterns across your corpus using regular expressions<br><a href="https://ladal.edu.au/string.html" style="color:#f0c060;font-size:.82rem;">&#x2192; See the full tutorial</a></div>
</div>


**What this tool does:** Strips unwanted content from your texts — XML tags, URLs, speaker labels, punctuation, numbers, or any custom pattern — and saves clean versions ready for further analysis.

**Steps:** Upload files → Select cleaning options → Run Analysis → Download cleaned texts


<div style="background:#51247a;color:white;padding:8px 16px;border-radius:6px;margin:1.2rem 0 .5rem 0;font-family:sans-serif;"><b>Step 1 &mdash; Set up the environment</b></div>


In [ ]:
suppressPackageStartupMessages(library(IRdisplay))
IRdisplay::display_html('
<style>
  .jp-InputArea-editor { display: none !important; }
  .jp-CodeCell .jp-Cell-inputWrapper { display: none !important; }
  .show-code .jp-CodeCell .jp-Cell-inputWrapper { display: flex !important; }
  .ladal-toolbar {
    position: sticky; top: 0; z-index: 999;
    background: #51247a; color: white;
    padding: 8px 18px; border-radius: 0 0 8px 8px;
    display: flex; align-items: center; gap: 14px;
    font-family: sans-serif; font-size: .85rem;
    box-shadow: 0 2px 8px rgba(81,36,122,.25);
  }
  .ladal-toolbar b { font-size: 1rem; }
  .ladal-toolbar button {
    background: rgba(255,255,255,.18); border: 1px solid rgba(255,255,255,.4);
    color: white; padding: 4px 12px; border-radius: 4px;
    cursor: pointer; font-size: .8rem; font-weight: 600;
  }
  .ladal-toolbar button:hover { background: rgba(255,255,255,.32); }
</style>
<div class="ladal-toolbar">
  <b>&#x1F4D3; LADAL Interactive Notebook</b>
  <button onclick="document.body.classList.toggle('show-code');
    this.textContent = document.body.classList.contains('show-code')
      ? 'Hide Code' : 'Show Code'">Show Code</button>
  <span style="opacity:.7;font-size:.78rem;">
    Code is hidden by default &mdash; click to toggle
  </span>
</div>
')


In [ ]:
# ── Suppress startup messages & load display helpers ────────────────
suppressPackageStartupMessages(library(IRdisplay))
suppressPackageStartupMessages(library(IRkernel))

# ── Colour-coded display helpers ────────────────────────────────────
.box <- function(msg, bg, border, icon="") {
  IRdisplay::display_html(paste0(
    '<div style="background:', bg, ';border-left:4px solid ', border,
    ';border-radius:6px;padding:11px 16px;margin:.6rem 0;',
    'font-family:sans-serif;font-size:.9rem;">',
    if (nzchar(icon)) paste0('<b>', icon, '</b> ') else '',
    msg, '</div>'))
}
ok   <- function(msg) .box(msg, "#eafaf1", "#27ae60", "&#x2705;")
warn <- function(msg) .box(msg, "#fff8e1", "#f0a500", "&#x26A0;&#xFE0F;")
err  <- function(msg) .box(msg, "#fff0f0", "#e74c3c", "&#x274C;")
info <- function(msg) .box(msg, "#f4f0f8", "#51247a", "&#x2139;&#xFE0F;")

# ── Progress bar helper ──────────────────────────────────────────────
.prog <- function(label, value, max=100) {
  pct <- round(100 * value / max)
  IRdisplay::display_html(paste0(
    '<div style="font-family:sans-serif;font-size:.85rem;margin:.4rem 0;">',
    '<span style="color:#51247a;font-weight:600;">', label, '</span><br>',
    '<div style="background:#e8e4f0;border-radius:4px;height:10px;margin-top:4px;">',
    '<div style="background:#51247a;width:', pct, '%;height:10px;',
    'border-radius:4px;transition:width .3s;"></div></div>',
    '<span style="color:#888;font-size:.78rem;">', pct, '%</span></div>'))
}

# ── Upload instructions ──────────────────────────────────────────────
upload_instructions <- function(folder="MyTexts", filetype="txt") {
  IRdisplay::display_html(paste0(
    '<div style="background:#f4f0f8;border:2px dashed #51247a;border-radius:8px;',
    'padding:18px 22px;margin:.8rem 0;font-family:sans-serif;">',
    '<b style="color:#51247a;font-size:1rem;">&#x1F4C2; Upload your files</b><br><br>',
    '<ol style="margin:0;padding-left:1.4em;font-size:.9rem;line-height:1.8;">',
    '<li>Find the <b>file browser panel</b> on the left side of the screen.</li>',
    '<li>Double-click the <b><code>', folder, '</code></b> folder to open it.</li>',
    '<li><b>Drag and drop</b> your <code>.', filetype, '</code> files into that folder.</li>',
    '<li>Come back here and click <b>Run Analysis</b> below.</li>',
    '</ol>',
    '<p style="margin:.6rem 0 0 0;font-size:.82rem;color:#888;">',
    'Only <code>.', filetype, '</code> files are accepted. ',
    'You can upload multiple files at once.</p></div>'))
}

# ── Load plain-text files ────────────────────────────────────────────
load_texts <- function(folder = "notebooks/MyTexts") {
  files <- list.files(folder, pattern = "\.txt$",
                      full.names = TRUE, recursive = FALSE)
  if (length(files) == 0)
    stop("No .txt files found in '", folder, "'. ",
         "Please upload files into the ", basename(folder), " folder.")
  texts <- vapply(files, function(f)
    paste(readLines(f, warn = FALSE, encoding = "UTF-8"), collapse = " "),
    character(1))
  names(texts) <- tools::file_path_sans_ext(basename(files))
  texts
}

# ── Save texts ───────────────────────────────────────────────────────
save_texts <- function(texts, folder = "notebooks/MyOutput") {
  dir.create(folder, showWarnings = FALSE, recursive = TRUE)
  for (nm in names(texts))
    writeLines(texts[[nm]], file.path(folder, paste0(nm, ".txt")))
}

# ── Save Excel ───────────────────────────────────────────────────────
save_excel <- function(df, filename, folder = "notebooks/MyOutput") {
  dir.create(folder, showWarnings = FALSE, recursive = TRUE)
  writexl::write_xlsx(as.data.frame(df), file.path(folder, filename))
}

ok("Environment ready. Scroll down to configure and run your analysis.")


<div style="background:#51247a;color:white;padding:8px 16px;border-radius:6px;margin:1.2rem 0 .5rem 0;font-family:sans-serif;"><b>Step 2 📂 &mdash; Upload your texts</b></div>


In [ ]:
upload_instructions("MyTexts", "txt")

<div style="background:#51247a;color:white;padding:8px 16px;border-radius:6px;margin:1.2rem 0 .5rem 0;font-family:sans-serif;"><b>Step 3 &mdash; Configure and run</b></div>


In [ ]:
suppressPackageStartupMessages(library(ipywidgets))

w_tags    <- ipywidgets::Checkbox(value=TRUE,  description="Remove XML/HTML tags  <tag>")
w_urls    <- ipywidgets::Checkbox(value=FALSE, description="Remove URLs")
w_emails  <- ipywidgets::Checkbox(value=FALSE, description="Remove email addresses")
w_nums    <- ipywidgets::Checkbox(value=FALSE, description="Remove numbers")
w_punct   <- ipywidgets::Checkbox(value=FALSE, description="Remove punctuation")
w_speaker <- ipywidgets::Checkbox(value=FALSE, description="Remove speaker labels  [SPEAKER_A]:")
w_lower   <- ipywidgets::Checkbox(value=FALSE, description="Convert to lowercase")
w_ws      <- ipywidgets::Checkbox(value=TRUE,  description="Collapse extra whitespace")
w_custom  <- ipywidgets::Textarea(value="", rows=3,
               description="Custom patterns (one per line, regex):",
               placeholder="e.g. \\bACT\\s+[IVX]+\\b",
               style=list(description_width="initial"), layout=list(width="450px"))
w_btn     <- ipywidgets::Button(description="  Run Analysis",
               button_style="primary", icon="eraser")
w_out     <- ipywidgets::Output()

ipywidgets::display(ipywidgets::VBox(list(
  ipywidgets::HTML("<b style='color:#51247a'>Select cleaning options:</b>"),
  ipywidgets::HBox(list(
    ipywidgets::VBox(list(w_tags, w_urls, w_emails, w_nums)),
    ipywidgets::VBox(list(w_punct, w_speaker, w_lower, w_ws))
  )),
  w_custom, w_btn, w_out
)))

ipywidgets::observe(w_btn, "clicks", function(change) {
  ipywidgets::with_output(w_out, {
    tryCatch({
      suppressPackageStartupMessages(library(stringr))
      .prog("Loading texts...", 10)
      texts <- load_texts("notebooks/MyTexts")
      ok(paste("Loaded", length(texts), "file(s)"))

      patterns <- character(0)
      if (w_tags$value)    patterns <- c(patterns, "<[^>]*>")
      if (w_urls$value)    patterns <- c(patterns, "https?://\\S+", "www\\.\\S+")
      if (w_emails$value)  patterns <- c(patterns, "[a-zA-Z0-9._%+\\-]+@[a-zA-Z0-9.\\-]+\\.[a-zA-Z]{2,}")
      if (w_nums$value)    patterns <- c(patterns, "\\d+")
      if (w_punct$value)   patterns <- c(patterns, "[^\\p{L}\\p{N}\\s]")
      if (w_speaker$value) patterns <- c(patterns, "(\\[[A-Z0-9_]+\\]\\s*:?|<[A-Z0-9$_\\-]+>\\s*)")

      custom_pats <- Filter(nzchar, trimws(strsplit(w_custom$value, "\n")[[1]]))
      patterns    <- c(patterns, custom_pats)

      .prog("Cleaning texts...", 50)
      clean_texts <- lapply(texts, function(txt) {
        for (pat in patterns) {
          txt <- tryCatch(stringr::str_remove_all(txt, pat),
                          error=function(e) { warn(paste("Invalid pattern:", pat)); txt })
        }
        if (w_lower$value) txt <- tolower(txt)
        if (w_ws$value)    txt <- stringr::str_squish(txt)
        txt
      })

      orig  <- sum(nchar(unlist(texts)))
      clean <- sum(nchar(unlist(clean_texts)))
      ok(paste0("Removed <b>", format(orig-clean, big.mark=","),
                " characters</b> (",
                round(100*(orig-clean)/orig, 1), "% of original)."))

      .prog("Saving files...", 90)
      save_texts(clean_texts)
      .prog("Done.", 100)
      ok(paste0("Saved <b>", length(clean_texts), " cleaned file(s)</b> to MyOutput folder."))
    }, error=function(e) err(conditionMessage(e)))
  })
})


---

### Citation

Schweinberger, Martin. (2025). *LADAL Text Cleaner*. Brisbane: The University of Queensland. <https://ladal.edu.au/tools.html>

```bibtex
@misc{schweinberger2025textcleaner,
  author = {Schweinberger, Martin},
  title  = {LADAL Text Cleaner},
  year   = {2025},
  organization = {The University of Queensland},
  url    = {https://ladal.edu.au/tools.html}
}
```


In [ ]:
sessionInfo()